In [3]:
%pip install geopandas pandas -q

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import geopandas as gpd
import pandas as pd

In [39]:
oak_zoning_gdf = gpd.read_file("data/raw/Zoning_20261007.geojson") #Oakland zoning data
mtc_sb79_zones_gdf = gpd.read_file("data/raw/mtc_sb79_tod_zones.geojson") #MTC SB79 tier 1 and 2 zone data
oak_parcels_gdf = gpd.read_file("data/raw/oak_parcels.geojson") #Oakland parcel data

In [40]:
#initial row count
initial_count = len(oak_parcels_gdf)

#dissolve duplicate APN polygons into a single multi-part geometry per APN so each row is one parcel
#APNs refer to specific units, not parcels, so this makes each row one parcel
oak_parcels_gdf = oak_parcels_gdf.dissolve(by="APN", as_index=False)

#new row count
final_count = len(oak_parcels_gdf)

#difference in row counts (number of lost rows)
dissolved_count = initial_count - final_count

print(f"Initial rows: {initial_count}")
print(f"Final rows:   {final_count}")
print(f"Dissolved:    {dissolved_count} duplicate rows merged into existing APNs")

Initial rows: 112966
Final rows:   112718
Dissolved:    248 duplicate rows merged into existing APNs


In [41]:
def convert_crs(gdf_change, gdf_base):
    if gdf_change.crs != gdf_base.crs: #check if input gdf mathches base gdf
        gdf_change = gdf_change.to_crs(gdf_base.crs) #if they don't match, change the crs to match
    return gdf_change

In [42]:
#make all gdfs the same crs as the mtc gdf
gdfs = [oak_zoning_gdf, oak_parcels_gdf]
for gdf in gdfs:
    convert_crs(gdf, mtc_sb79_zones_gdf)

In [43]:
keep_parcel_columns = [
    'APN', #Assessor's Parcel Number
    'APN_SORT', #Assessor's Parcel Number without dashes
    'CENTROID_X', 
    'CENTROID_Y', 
    'Land', #Assessed land value (excluding structure value)
    'Imps', #Value of improvements (structure/infrastructure) on the land
    'UseCode', #Current land use (not what's zoned)
    'SitusAddress', #Address
    'geometry'
]
oak_parcels_gdf = oak_parcels_gdf[keep_parcel_columns] #Keep only the columns necessary to this project to improve processeing speeds

In [44]:
#chnage centroid column names. They are the x and y coordinates of each parcels centroid
oak_parcels_gdf = oak_parcels_gdf.rename(
    columns={"CENTROID_X": "parcel_centroid_x", "CENTROID_Y": "parcel_centroid_y"}
)

In [59]:
def get_dominant_intersect(parcels_gdf, attribute_gdf):
    
    #create a temportary unique ID in case of duplicate parcels
    gdf = parcels_gdf.copy() #ensure not modifying parcels_gdf in-place
    gdf['temp_id'] = gdf.index

    before_count = len(parcels_gdf)

    #overaly the parcels with the attribute gdf their intersection
    intersected = gpd.overlay(gdf, attribute_gdf, how="intersection")
    
    duplicated_count = len(intersected)

    #get the area of each parcel - reproject to CA State Place Zone 3 (EPSG: 2227)
    #there are now duplicate parcels because they were assigned to multiple zones
    intersected["overlap_area"] = intersected.to_crs(epsg=2227).area

    #take only the parcel with the larger area in the SB79 zone
    dominant_intersect = (
        intersected.sort_values("overlap_area", ascending=False)
        .drop_duplicates(subset=["temp_id"])
    )

    matched_count = len(dominant_intersect)
    unmatched_count = before_count - matched_count
    
    # Print statement breakdown
    print(f"Initial parcels input:                 {before_count}")
    print(f"Total overlapping fragments created:    {duplicated_count}")
    print(f"Unique parcels matching attribute:     {matched_count}")
    print(f"Parcels with NO match (dropped):       {unmatched_count}\n")

    attr_cols = [
        col for col in dominant_intersect.columns 
        if col not in parcels_gdf.columns or col == "temp_id"
    ]
    
    return dominant_intersect[attr_cols]

In [60]:
oak_parcels = oak_parcels_gdf.copy()

#set the temp_id to compare the clipped versions to
oak_parcels["temp_id"] = oak_parcels.index 

In [61]:
#overlay the parcels on the intersection of the SB79 zones and the Oakland zoning geometry
oak_parcel_tiers = get_dominant_intersect(oak_parcels, mtc_sb79_zones_gdf)
oak_parcel_zones = get_dominant_intersect(oak_parcels, oak_zoning_gdf)

Initial parcels input:                 112718
Total overlapping fragments created:    39573
Unique parcels matching attribute:     35878
Parcels with NO match (dropped):       76840

Initial parcels input:                 112718
Total overlapping fragments created:    133573
Unique parcels matching attribute:     112680
Parcels with NO match (dropped):       38



In [62]:
initial_count = len(oak_parcels)

#merge original parcels with parcel tiers
oak_parcels = oak_parcels.merge(
    oak_parcel_tiers,
    on="temp_id",
    how="inner" #drop parcels that don't have a tier match
)

tier_merge_count = len(oak_parcels)
print(f"Rows after merging SB79 Tiers:      {tier_merge_count}")
print(f"Change: {tier_merge_count - initial_count} rows")

#prevent duplicate column names
oak_parcels = oak_parcels.rename(columns={
    "zone_id": "tier_zone_id", 
    "zone_label": "tier_zone_label"
})

#keep only 3 relevant columns from the zonign data and temp_id to merge
oak_parcels = oak_parcels.merge(
    oak_parcel_zones[["temp_id", "basezone", "overlay", "znlabel"]],
    on="temp_id",
    how="left" #keep all the parcels
)

zoning_merge_count = len(oak_parcels)
print(f"Rows after merging Oakland Zoning:  {zoning_merge_count}")
print(f"Change: {zoning_merge_count - tier_merge_count} rows")
print(f"Final Parcel count: {len(oak_parcels)}")

#drop temp_id because clipping and merging is done
oak_parcels = oak_parcels.drop(columns=["temp_id"])

Rows after merging SB79 Tiers:      35878
Change: -76840 rows
Rows after merging Oakland Zoning:  35878
Change: 0 rows
Final Parcel count: 35878


In [63]:
#save cleaned and joined gdf as a parqet file to use for analysis
oak_parcels.to_parquet("data/processed/oak_parcels_sb79tiers_zones.parquet")